### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")


llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022DDF0F1760>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022DDF0F3530>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  

In [9]:
## VecotrStore
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embeddings)
vectorstore

In [15]:
# Bloqueia a execução até obter o resultado
# Executa sequencialmente
# Mais simples de usar, mas pode travare aplicações se a busca demorar
# Ideal para scripts simples ou quando você quer esperar o resultado
vectorstore.similarity_search("cat")

[Document(id='1e065410-0e48-4569-9209-74e081f9d60f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='0947d523-b87c-45b9-82ee-4233981f3629', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='2e5859b4-937d-4cf6-9be2-610b09637858', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='fbd2c8d0-bcdd-44b8-96b5-508a18bf323e', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [16]:
# Async query
# Não bloqueia a execução
# Permite que outras operações rodem enquanto aguarda o resultado
# Usa corrotinas (async/await do Python)
# Melhor para aplicações que precisam lidar com múltiplas requisições simultaneamente
await vectorstore.asimilarity_search("cat")

[Document(id='1e065410-0e48-4569-9209-74e081f9d60f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='0947d523-b87c-45b9-82ee-4233981f3629', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='2e5859b4-937d-4cf6-9be2-610b09637858', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='fbd2c8d0-bcdd-44b8-96b5-508a18bf323e', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

Quando usar cada um entre Sincrono e Assincrono:

Síncrono: Scripts simples, prototipagem, quando a performance não é crítica

Assíncrono: APIs com múltiplos usuários, aplicações web, quando precisa de melhor concorrência

In [14]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='1e065410-0e48-4569-9209-74e081f9d60f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351058006286621),
 (Document(id='0947d523-b87c-45b9-82ee-4233981f3629', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740898847579956),
 (Document(id='2e5859b4-937d-4cf6-9be2-610b09637858', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
  1.5956902503967285),
 (Document(id='fbd2c8d0-bcdd-44b8-96b5-508a18bf323e', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.66579270362854)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

Vector Store:

- Apenas uma estrutura de dados/base de dados
- Não é um "Runnable" (não segue o padrão LCEL)
- Você chama métodos diretamente como similarity_search()
- Não pode ser facilmente combinado em pipelines
- 
Retriever:

- É um Runnable (segue o padrão LCEL)
- Implementa métodos padronizados (invoke, invoke_async, batch, etc.)
- Pode ser integrado em chains e pipelines
- Reutiliza métodos do Vector Store por baixo

Basicamente, um Retriever é a forma correta e moderna de usar Vector Stores dentro do ecosistema LangChain, porque segue um padrão consistente que permite composição e reutilização em pipelines complexos.

In [19]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat", "dog"])

[[Document(id='1e065410-0e48-4569-9209-74e081f9d60f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='0947d523-b87c-45b9-82ee-4233981f3629', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [21]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriever.batch(["cat", "dog"])

[[Document(id='1e065410-0e48-4569-9209-74e081f9d60f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='0947d523-b87c-45b9-82ee-4233981f3629', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [ ]:
## This is a basic RAG 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", message)
    ]
)

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("Tell me about dogs")
print(response.content)

Dogs are great companions, known for their loyalty and friendliness.


O retriever é a informação do nosso vector store que servira de contexto!